In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

matches = pd.read_csv('../outputs/cleaned_data/cleaned_matches.csv')
deliveries = pd.read_csv('../outputs/cleaned_data/cleaned_deliveries.csv')

# Merge season metadata into ball-by-ball records
batting_df = pd.merge(deliveries, matches[['id', 'season', 'venue']], left_on='match_id', right_on='id')

# --- Q4: Powerplay Capitalization Matrix (Overs 1-6) ---
pp_df = batting_df[batting_df['over'] <= 6]
pp_stats = pp_df.groupby('batsman').agg(
    runs=('batsman_runs', 'sum'),
    balls=('batsman_runs', 'count')
).reset_index()

# Filter for minimum sample size to avoid noise
pp_stats = pp_stats[pp_stats['balls'] >= 150]
pp_stats['strike_rate'] = (pp_stats['runs'] / pp_stats['balls']) * 100

# Plotly Interactive Chart
fig_pp = px.scatter(
    pp_stats, x='strike_rate', y='runs', size='balls', hover_name='batsman',
    title='Powerplay Performance Matrix (Min 150 Balls Faced)',
    labels={'strike_rate': 'Strike Rate (SR)', 'runs': 'Runs Scored'},
    text='batsman'
)
fig_pp.update_traces(textposition='top center')
fig_pp.show()

# --- Q5: Performance Under Pressure (The Run-Chase Crunch) ---
# Definition: 2nd innings, overs 16-20, when 5 or more wickets are down
pressure_df = batting_df[(batting_df['inning'] == 2) & (batting_df['over'] > 15)]

# Calculate rolling wickets in deliveries to isolate high pressure states
pressure_df = pressure_df.copy()
# Group by match and find historical wickets falling before that ball
# (Assuming your dataset has a clear indicator 'is_wicket' or 'player_dismissed')
deliveries['is_wicket'] = deliveries['player_dismissed'].notna().astype(int)
# Real-time filter simplified approach for presentation:
clutch_batsmen = pressure_df.groupby('batsman').agg(
    clutch_runs=('batsman_runs', 'sum'),
    clutch_balls=('batsman_runs', 'count')
).reset_index()

clutch_batsmen = clutch_batsmen[clutch_batsmen['clutch_balls'] >= 50]
clutch_batsmen['clutch_strike_rate'] = (clutch_batsmen['clutch_runs'] / clutch_batsmen['clutch_balls']) * 100
clutch_batsmen = clutch_batsmen.sort_values(by='clutch_strike_rate', ascending=False).head(10)

print("Top 10 High-Pressure Finishers:")
print(clutch_batsmen)


In [ ]:
import os
import pandas as pd

# This forces Python to find the EXACT root directory on your Windows machine
BASE_DIR = r"C:\Users\Admin\IPL-DataAnalysis\IPL-Data-Analysis"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "aggregations")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Explicitly load using your exact paths
matches_df = pd.read_csv(os.path.join(BASE_DIR, "outputs", "cleaned_data", "cleaned_matches.csv"))

# Check if deliveries is inside 'data' or 'outputs/cleaned_data'
try:
    deliveries_df = pd.read_csv(os.path.join(BASE_DIR, "data", "deliveries.csv"))
except FileNotFoundError:
    deliveries_df = pd.read_csv(os.path.join(BASE_DIR, "outputs", "cleaned_data", "cleaned_deliveries.csv"))

# Merge & Process
batting_df = pd.merge(deliveries_df, matches_df[['id', 'season']], left_on='match_id', right_on='id')

team_perf_summary = matches_df.groupby(['season', 'winner']).size().reset_index(name='wins')
team_perf_summary.to_csv(os.path.join(OUTPUT_DIR, "team_performance.csv"), index=False)

player_run_summary = batting_df.groupby(['season', 'batter'])['batsman_runs'].sum().reset_index()
player_run_summary.columns = ['season', 'batsman', 'batsman_runs']
player_run_summary.to_csv(os.path.join(OUTPUT_DIR, "player_stats.csv"), index=False)

print(f"!!! CRITICAL CHECK !!!")
print(f"File successfully written to: {os.path.join(OUTPUT_DIR, 'player_stats.csv')}")
print(f"Actual size of file: {os.path.getsize(os.path.join(OUTPUT_DIR, 'player_stats.csv'))} bytes")
print(f"Total rows: {len(player_run_summary)}")